In [ ]:

"""
intput: video file
output:
#  Data Adapter ：
pose_data_payload = {
    0: { # Frame 0
        "right_shoulder": {"x": 0.51, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.60, "y": 0.35, "z": 0.45},
        "right_wrist": {"x": 0.65, "y": 0.30, "z": 0.50},
        "right_hip": {"x": 0.50, "y": 0.80, "z": 0.50}
    },
    1: { # Frame 1
        "right_shoulder": {"x": 0.52, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.65, "y": 0.20, "z": 0.55},
        "right_wrist": {"x": 0.75, "y": 0.05, "z": 0.60},
        "right_hip": {"x": 0.51, "y": 0.80, "z": 0.50}
    },
    # ...
}
"""
from typing import Dict, Any
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os
import sys

current_dir = os.path.dirname(os.path.abspath(__file__))
project_root = os.path.dirname(os.path.join(current_dir, "../"))
sys.path.append(project_root)

from config.core import config

MODEL_PATH = config.get_path("POSE_MODEL_PATH")
VIDEO_PATH = config.get_path("VIDEO_PATH")

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)
# Create a PoseLandmarker object
detector = vision.PoseLandmarker.create_from_options(options)

# init the pose data payload
pose_data_payload: Dict[int, Any] = {}

# Open the webcam
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Frames per second: {fps}")
# Process the video frames
success, frame = cap.read()
if success:
    print(f"success: {success}, frame shape: {frame.shape}")
    # Convert the BGR image to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    print(f"converted successfully, frame_rgb shape: {frame_rgb.shape}")
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    print(f"mp_image created successfully, mp_image shape: {mp_image.width}x{mp_image.height}")
    detection_result = detector.detect_for_video(mp_image, timestamp_ms=0)
    # ensure the detection result contains pose landmarks
    if detection_result.pose_landmarks:
        print("Pose landmarks detected:")

        # Extract all the 33 points
        landmarks = detection_result.pose_landmarks[0]

        current_frame_data = {
            "right_shoulder": {"x": landmarks[12].x, "y": landmarks[12].y, "z": landmarks[12].z},
            "right_elbow": {"x": landmarks[14].x, "y": landmarks[14].y, "z": landmarks[14].z},
            "right_wrist": {"x": landmarks[16].x, "y": landmarks[16].y, "z": landmarks[16].z},
            "right_hip": {"x": landmarks[24].x, "y": landmarks[24].y, "z": landmarks[24].z},
        }
        pose_data_payload[0] = current_frame_data
        print("--- Frame 0 Pose Data ---")
        print(f"Right Shoulder: {current_frame_data['right_shoulder']}")
    else:
        print("No pose landmarks detected.")
else:
    print("Failed to read the video frame.")

cap.release()
detector.close()



In [ ]:

"""
intput: video file
output:
#  Data Adapter ：
pose_data_payload = {
    0: { # Frame 0
        "right_shoulder": {"x": 0.51, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.60, "y": 0.35, "z": 0.45},
        "right_wrist": {"x": 0.65, "y": 0.30, "z": 0.50},
        "right_hip": {"x": 0.50, "y": 0.80, "z": 0.50}
    },
    1: { # Frame 1
        "right_shoulder": {"x": 0.52, "y": 0.50, "z": 0.50},
        "right_elbow": {"x": 0.65, "y": 0.20, "z": 0.55},
        "right_wrist": {"x": 0.75, "y": 0.05, "z": 0.60},
        "right_hip": {"x": 0.51, "y": 0.80, "z": 0.50}
    },
    # ...
}
"""
from typing import Any
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

MODEL_PATH = '../models/pose_landmarker_heavy.task'
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False
)
# Create a PoseLandmarker object
detector = vision.PoseLandmarker.create_from_options(options)

# init the pose data payload
pose_data_payload: dict[int, Any] = {}
frame_idx = 0
# Open the webcam
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Frames per second: {fps}")
while True:
    # Process the video frames
    success, frame = cap.read()
    if not success:
        print("End of video reached or failed to read the video frame.")
        break

    # Convert the BGR image to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    # set up for timestamp in milliseconds for the current frame
    timestamp_ms = int((frame_idx / fps) * 1000)

    detection_result = detector.detect_for_video(mp_image, timestamp_ms=timestamp_ms)
    # ensure the detection result contains pose landmarks
    if detection_result.pose_landmarks:
        print("Pose landmarks detected:")

        # Extract all the 33 points
        # Note: detection_result.pose_landmarks is a list of PoseLandmarkList, where each PoseLandmarkList corresponds to a detected person in the frame. For simplicity, we will only consider the first detected person (if multiple people are detected).
        landmarks = detection_result.pose_landmarks[0]

        current_frame_data = {
            "right_shoulder": {"x": landmarks[12].x, "y": landmarks[12].y, "z": landmarks[12].z},
            "right_elbow": {"x": landmarks[14].x, "y": landmarks[14].y, "z": landmarks[14].z},
            "right_wrist": {"x": landmarks[16].x, "y": landmarks[16].y, "z": landmarks[16].z},
            "right_hip": {"x": landmarks[24].x, "y": landmarks[24].y, "z": landmarks[24].z},
        }
        pose_data_payload[frame_idx] = current_frame_data
        print(f"--- Frame {frame_idx} Pose Data ---")
        print(f"Right Shoulder: {current_frame_data['right_shoulder']}")
        print(f"Right Elbow: {current_frame_data['right_elbow']}")
        print(f"Right Wrist: {current_frame_data['right_wrist']}")
        print(f"Right Hip: {current_frame_data['right_hip']}")

    else:
        pose_data_payload[frame_idx] = None
        print("No pose landmarks detected.")
    frame_idx += 1

cap.release()
detector.close()


In [ ]:

from typing import Any
import matplotlib.pyplot as plt


def calculate_joint_velocity(pose_data_payload: dict[int, Any], joint_name: str) -> list:
    velocities = []
    joint_x = []
    joint_y = []
    joint_z = []

    for i in pose_data_payload.keys():
        joint_x.append(pose_data_payload[i][f"{joint_name}"]["x"])
        joint_y.append(pose_data_payload[i][f"{joint_name}"]["y"])
        joint_z.append(pose_data_payload[i][f"{joint_name}"]["z"])

    for i in range(1, len(joint_x)):
        dx = joint_x[i] - joint_x[i - 1]
        dy = joint_y[i] - joint_y[i - 1]
        dz = joint_z[i] - joint_z[i - 1]
        velocity = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)
        velocities.append(velocity)
    return velocities


joint_name = ["right_shoulder", "right_elbow", "right_wrist"]
plt.figure()

for joint in joint_name:
    # Display the velocities

    plt.plot(calculate_joint_velocity(pose_data_payload, joint), label=f'{joint} Velocity')

plt.title(f'joint Velocity Over Time')
plt.xlabel('Frame')
plt.ylabel('Velocity')
plt.grid(True)
plt.show()






In [ ]:
from typing import Any
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter


def calculate_joint_velocity(pose_data_payload: dict[int, Any], joint_name: str) -> list:
    velocities = []
    joint_x = []
    joint_y = []
    joint_z = []

    for i in pose_data_payload.keys():
        joint_x.append(pose_data_payload[i][f"{joint_name}"]["x"])
        joint_y.append(pose_data_payload[i][f"{joint_name}"]["y"])
        joint_z.append(pose_data_payload[i][f"{joint_name}"]["z"])

    for i in range(1, len(joint_x)):
        dx = joint_x[i] - joint_x[i - 1]
        dy = joint_y[i] - joint_y[i - 1]
        dz = joint_z[i] - joint_z[i - 1]
        velocity = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)
        velocities.append(velocity)
    return velocities


joint_name = ["right_shoulder", "right_elbow", "right_wrist"]
plt.figure()

for joint in joint_name:
    # Display the velocities
    smooth_velocity = savgol_filter(calculate_joint_velocity(pose_data_payload, joint), window_length=11, polyorder=3)
    print(smooth_velocity)
    plt.plot(smooth_velocity, label=f'{joint} Velocity')

plt.title(f'joint Velocity Over Time')
plt.xlabel('Frame')
plt.ylabel('Velocity')
plt.grid(True)
plt.show()






In [ ]:
from typing import Any
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, medfilt


def calculate_joint_velocity(pose_data_payload: dict[int, Any], joint_name: str) -> list:
    """"
    Calculate the velocity of a specific joint across frames.
    Args:
        pose_data_payload (dict[int, Any]): The pose data payload containing joint positions for each frame.
        joint_name (str): The name of the joint for which to calculate velocity.
    Returns:
        list: A list of velocities for the specified joint across frames.
    """
    velocities = []
    joint_x = []
    joint_y = []
    joint_z = []

    for i in pose_data_payload.keys():
        joint_x.append(pose_data_payload[i][f"{joint_name}"]["x"])
        joint_y.append(pose_data_payload[i][f"{joint_name}"]["y"])
        joint_z.append(pose_data_payload[i][f"{joint_name}"]["z"])

    for i in range(1, len(joint_x)):
        dx = joint_x[i] - joint_x[i - 1]
        dy = joint_y[i] - joint_y[i - 1]
        dz = joint_z[i] - joint_z[i - 1]
        velocity = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)
        velocities.append(velocity)
    return velocities


joint_name = ["right_shoulder", "right_elbow", "right_wrist"]
plt.figure()


def smooth_data_processing(pose_data_payload: dict[int, Any], joint_name: str, plot_axis: int,
                           window_length: int = 9, polyorder=3) -> None:
    """
    Process the raw joint velocity data by applying a smoothing filter and plot the results.
    Args:        pose_data_payload (dict[int, Any]): The pose data payload containing joint positions for each frame
        joint_name (str): The name of the joint for which to calculate velocity.
        plot_axis (int): The index of the subplot axis to plot on.
        window_length (int, optional): The length of the filter window (must be a positive odd integer). Defaults to 9.
        polyorder (int, optional): The order of the polynomial used to fit the samples. Defaults to 3.
        Returns:
        None: This function does not return anything, it directly plots the results on the specified axis.
    """
    raw_joint_vel = calculate_joint_velocity(pose_data_payload, joint_name)
    clean_vel = medfilt(raw_joint_vel, kernel_size=5)
    smooth_joint_vel = savgol_filter(clean_vel, window_length=window_length, polyorder=polyorder)
    axes[plot_axis].plot(raw_joint_vel, alpha=0.3, label="Raw", color='red')
    axes[plot_axis].plot(smooth_joint_vel, label="Smoothed", color='green')
    axes[plot_axis].set_title(f"{joint_name} Velocity")
    axes[plot_axis].legend()


fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 8), sharex=True)

window_length = 9
polyorder = 5
for idx, joint in enumerate(joint_name):
    smooth_data_processing(pose_data_payload, joint, idx, window_length, polyorder)






In [ ]:
from moviepy import VideoFileClip
from scipy.signal import find_peaks
import librosa
import os

video = VideoFileClip(VIDEO_PATH)
audio_path = "../data/audio/temp_audio.wav"
# Extract audio from the video and save it as a temporary file
video.audio.write_audiofile(audio_path, logger=None)
# Load the audio file using librosa
y, sr = librosa.load(audio_path, sr=None)
# Now you can use the audio data (y) and sample rate (sr) for further processing
print(f"Audio loaded successfully, sample rate: {sr}, audio shape: {y}")
# detect the onset strength of the impact sound
onset_env = librosa.onset.onset_strength(y=y, sr=sr)
audio_peaks_frames = librosa.util.peak_pick(
    onset_env,
    # Number of frames before and after the current frame to consider for peak picking
    pre_max=3, post_max=3,
    # Number of frames before and after the current frame to consider for calculating the average onset strength
    pre_avg=3, post_avg=5,  # before impact sound, it's quiet, after impact sound, there might be noise or echo
    delta=0.5,  # Threshold for peak picking. A higher value means that only stronger peaks will be detected.
    wait=10  # Minimum number of frames between detected peaks. Set to 0 to

)
audio_times_seconds = librosa.frames_to_time(audio_peaks_frames, sr=sr)
print(fps)
audio_video_frames = [int(t * fps) for t in
                      audio_times_seconds]  # Convert audio peak times to corresponding video frame indices
print("--------------")
print(audio_video_frames)

if os.path.exists(audio_path):
    # Remove the temporary audio file after loading it
    os.remove(audio_path)
    print("Temporary audio file removed.")

# analyze the kinetic chain( the change in the wrist velocity) to cross-validate the detected impact sound peaks
raw_right_wrist_vel = calculate_joint_velocity(pose_data_payload, "right_wrist")
clean_vel = medfilt(raw_right_wrist_vel, kernel_size=5)
right_wrist_vel = savgol_filter(clean_vel, window_length=window_length, polyorder=polyorder)
print("----------")
print(right_wrist_vel)
try:
    visual_peaks, _ = find_peaks(right_wrist_vel, height=np.max(right_wrist_vel) * 0.4, distance=10
                                 # Minimum number of frames between detected peaks in the velocity data, to avoid detecting multiple peaks for a single impact event
                                 )
    print(list(visual_peaks))
    # Cross-validate the detected audio peaks with the visual peaks from the wrist velocity data
    TOLERANCE = 2  # Number of frames within which to consider an audio peak and a visual peak as matching
    confirmed_impacts = []
    for audio_peak in audio_video_frames:
        for visual_peak in visual_peaks:
            if abs(audio_peak - visual_peak) <= TOLERANCE:
                final_frame = int((audio_peak + visual_peak) / 2)
                confirmed_impacts.append(final_frame)
                break
    if confirmed_impacts:
        final_impact = confirmed_impacts[0]
        print(
            f"Confirmed impact detected at video frame: {final_impact}, which corresponds to time: {final_impact / fps:.2f} seconds")
    else:
        # If no confirmed impacts are found, we can still use the visual peaks to determine the most likely impact frame based on the highest wrist velocity
        final_impact = visual_peaks[int(np.argmax([right_wrist_vel[smooth_idx] for smooth_idx in visual_peaks]))]
        print(
            f"No confirmed impacts, but the most likely impact frame based on wrist velocity is: {final_impact}, which corresponds to time: {final_impact / fps:.2f} seconds")
except TypeError as e:
    print(f"Error in find_peaks: {e}")
except Exception as e:
    print(f"Unexpected error in find_peaks: {e}")







In [ ]:
import matplotlib.pyplot as plt

# 假设 onset_env 是你算出来的起音强度，peaks 是你找出的点
plt.figure(figsize=(12, 4))
plt.plot(onset_env, label='Onset Strength (Audio Energy)', color='gray')
plt.plot(audio_peaks_frames, onset_env[audio_peaks_frames], 'rx', markersize=10, label='Detected Impacts')
plt.axhline(0, color='black', linewidth=0.5)
plt.title("Audio Peak Detection Debugger")
plt.xlabel("Audio Frames (~11.6ms per frame)")
plt.ylabel("Energy")
plt.legend()
plt.show()

In [ ]:
from typing import Any
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, medfilt

def calculate_joint_velocity(
        vx: list[np.ndarray],
        vy: list[np.ndarray],
        vz: list[np.ndarray]
) -> list:
    """
    Calculates the 3D scalar speed from independent X, Y, Z velocity components.

    Args:
        vx: 1D array of velocity components along the X axis.
        vy: 1D array of velocity components along the Y axis.
        vz: 1D array of velocity components along the Z axis.

    Returns:
        list[float]: A list of scalar speeds for each frame, rounded to 3 decimal places.
    """
    vel_vectors = np.stack([vx, vy, vz], axis=1)
    scalar_speeds = np.linalg.norm(vel_vectors, axis=1)
    return np.round(scalar_speeds, 3).tolist()


def extract_coords(pose_data: dict, joint_name: str, coords_idx: str) -> list:
    """
    Extract the 1D coordinates
    """
    return [pose_data[frame][joint_name][coords_idx] for frame in sorted(pose_data.keys())]


def calculate_smoothed_vel(
        coordinates: list,
        fps: float,
        window_length: int = 5,
        polyorder: int = 2,
        deriv: int = 1  # Time difference
) -> np.ndarray:
    """
    Applies a Savitzky-Golay filter to smooth a 1D time series and calculate its derivative.

    Args:
        coordinates: Raw 1D coordinate array (e.g., just the X axis over time).
        window_length: The length of the filter window (must be an odd integer).
        polyorder: The order of the polynomial used to fit the samples.
        deriv: The order of the derivative to compute (1 for First Derivative).

    Returns:
        np.ndarray: The smoothed derivative array (representing 1D velocity).
    """
    delta = 1 / fps
    return savgol_filter(coordinates, window_length, polyorder, deriv, delta)


def extract_joint_velocity(pose_data: dict, joint_name: str, fps: float,window_length) -> list[float]:
    """
    Extract the velocity of a joint across frames during the impact event.
    param joint_name: The name of the joint to extract velocity for (e.g., "right_shoulder", "right_elbow", "right_wrist")
    param pose_data: The pose data containing 3D coordinates for each body part across frames
    return: A list of velocities for the specified joint across frames
    """
    joint_positions_x: list = extract_coords(pose_data, joint_name, 'x')
    joint_positions_y: list = extract_coords(pose_data, joint_name, 'y')
    joint_positions_z: list = extract_coords(pose_data, joint_name, 'z')
    vx = calculate_smoothed_vel(joint_positions_x, fps,window_length)
    vy = calculate_smoothed_vel(joint_positions_y, fps,window_length)
    vz = calculate_smoothed_vel(joint_positions_z, fps,window_length)
    velocities: list = calculate_joint_velocity(vx, vy, vz)

    return velocities


def plot_smoothed_data(pose_data_payload, joint_name, plot_axis,window_length):
    velocity = extract_joint_velocity(pose_data_payload, joint_name, 60,window_length)
    raw_joint_vel = calculate_joint_velocity_after(pose_data_payload, joint_name)
    axes[plot_axis].plot(raw_joint_vel, alpha=0.3, label="Raw", color='red')
    axes[plot_axis].plot(velocity, label="Smoothed filter on X,Y,Z -> V", color='blue')
    axes[plot_axis].set_title(f"{joint_name} Velocity")
    axes[plot_axis].legend()

def calculate_joint_velocity_after(pose_data_payload: dict[int, Any], joint_name: str) -> list:
    """"
    Calculate the velocity of a specific joint across frames.
    Args:
        pose_data_payload (dict[int, Any]): The pose data payload containing joint positions for each frame.
        joint_name (str): The name of the joint for which to calculate velocity.
    Returns:
        list: A list of velocities for the specified joint across frames.
    """
    velocities = []
    joint_x = []
    joint_y = []
    joint_z = []

    for i in pose_data_payload.keys():
        joint_x.append(pose_data_payload[i][f"{joint_name}"]["x"])
        joint_y.append(pose_data_payload[i][f"{joint_name}"]["y"])
        joint_z.append(pose_data_payload[i][f"{joint_name}"]["z"])

    for i in range(1, len(joint_x)):
        dx = joint_x[i] - joint_x[i - 1]
        dy = joint_y[i] - joint_y[i - 1]
        dz = joint_z[i] - joint_z[i - 1]
        velocity = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)*60
        velocities.append(velocity)
    return velocities


joint_name = ["right_shoulder", "right_elbow", "right_wrist"]
plt.figure()


def smooth_data_processing(pose_data_payload: dict[int, Any], joint_name: str, plot_axis: int,
                           window_length: int = 5, polyorder=3) -> None:
    """
    Process the raw joint velocity data by applying a smoothing filter and plot the results.
    Args:        pose_data_payload (dict[int, Any]): The pose data payload containing joint positions for each frame
        joint_name (str): The name of the joint for which to calculate velocity.
        plot_axis (int): The index of the subplot axis to plot on.
        window_length (int, optional): The length of the filter window (must be a positive odd integer). Defaults to 9.
        polyorder (int, optional): The order of the polynomial used to fit the samples. Defaults to 3.
        Returns:
        None: This function does not return anything, it directly plots the results on the specified axis.
    """
    raw_joint_vel = calculate_joint_velocity_after(pose_data_payload, joint_name)
    clean_vel = medfilt(raw_joint_vel, kernel_size=5)
    smooth_joint_vel = savgol_filter(clean_vel, window_length=window_length, polyorder=polyorder)
    axes[plot_axis].plot(raw_joint_vel, alpha=0.3, label="Raw", color='red')
    axes[plot_axis].plot(smooth_joint_vel, label="Smoothed filter on V", color='green')
    axes[plot_axis].set_title(f"{joint_name} Velocity")
    axes[plot_axis].legend()


fig, axes = plt.subplots(nrows=6, ncols=1, figsize=(10, 8), sharex=True)

window_length = 9
polyorder = 5
for idx, joint in enumerate(joint_name):
    smooth_data_processing(pose_data_payload, joint, idx, window_length, polyorder)
    plot_smoothed_data(pose_data_payload, joint, idx + 3,window_length)







